# 07장 보안 실습 — 계정·권한·GTFOBins 검토


## Goal

UID 0·특수 비트·쓰기 권한을 정상 기준선과 비교하고 GTFOBins 관련성·설정·실행 근거를 분리합니다.

[교안과 분석 질문](../../07-secure-scripting/07-3-account-permission-review.md)을 먼저 읽습니다.


## Setup

Python 커널의 %%bash를 사용합니다. 새 임시 폴더에 합성 자료와 결과 경로를 준비합니다. 외부 접속·서비스 등록·원본 서버 조사는 하지 않습니다. 코드를 검토하고 Setup부터 순서대로 실행합니다. Bash 셀 사이의 상태는 환경 변수와 파일로 전달합니다.


In [ ]:
from pathlib import Path
import hashlib
import os
import tempfile

lab = Path(tempfile.mkdtemp(prefix='bash-security-07-'))
data = lab / 'data'
output = lab / 'output'
data.mkdir()
output.mkdir()
fixtures = {'passwd.sample': 'root:x:0:0:root:/root:/bin/bash\nanalyst:x:1000:1000:Analyst:/home/analyst:/bin/bash\ncollector:x:995:995:Collector:/var/lib/collector:/usr/sbin/nologin\nlegacy-admin:x:0:0:Legacy:/var/lib/legacy:/usr/sbin/nologin\n', 'permissions.psv': 'path|owner|group|mode|purpose|approval\n/usr/bin/passwd|root|root|4755|password-management|baseline\n/opt/collector/bin/report|root|collector|0775|service-executable|review\n/var/tmp/course-cache|root|root|1777|shared-temp|baseline\n', 'tool-review.psv': 'case_id|tool_role|reference_listed|context|business_need|approval|scope_fit|telemetry\nR01|text-filter|yes|unprivileged|documented|approved|aligned|present\nR02|report-helper|yes|sudo|unknown|unknown|unknown|not_collected\nR03|custom-helper|no|service|documented|unknown|review|not_collected\nR04|network-helper|yes|capabilities|documented|approved|aligned|present\n'}
for name, content in fixtures.items():
    path = data / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
before = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
tools_dir = lab / 'tools'
tools_dir.mkdir()
os.environ['COURSE_TOOLS'] = str(tools_dir)
os.environ['COURSE_DATA'] = str(data)
os.environ['COURSE_OUT'] = str(output)
print('합성 자료와 새 결과 폴더 준비 완료')


## Steps

예상 결과: UID 0 계정 2개, 기준선 SUID 1개, 검토 항목 1개; 별도 카드 aligned=2/review=1/unknown=1, 악용 입증 아님

명령을 실행하기 전에 입력·출력·실패 조건을 표시합니다. 자료의 상세 필드 해석과 정상 행위 대안은 연결된 교안에서 확인합니다.


### 1. UID 0 계정과 로그인 셸 구분


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
awk -F: '$3==0 {print $1 "|" $7}' "$COURSE_DATA/passwd.sample" > "$COURSE_OUT/uid0.psv"
test "$(wc -l < "$COURSE_OUT/uid0.psv")" -eq 2
grep -Fx 'legacy-admin|/usr/sbin/nologin' "$COURSE_OUT/uid0.psv"


### 2. 비트와 승인 기준선 함께 읽기


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
awk -F '|' 'NR>1 && $4 ~ /^[4567]/ {print $1 "|" $4 "|" $6}' \
 "$COURSE_DATA/permissions.psv" > "$COURSE_OUT/suid-review.psv"
grep -Fx '/usr/bin/passwd|4755|baseline' "$COURSE_OUT/suid-review.psv"


### 3. 검토 대상과 취약점 확정 구분


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
awk -F '|' 'NR>1 && $6=="review" {print $1 "|" $3 "|" $4}' \
 "$COURSE_DATA/permissions.psv" > "$COURSE_OUT/review.psv"
grep -Fx '/opt/collector/bin/report|collector|0775' "$COURSE_OUT/review.psv"
printf 'review_items=1 exploitation_proven=no\n'


### 4. GTFOBins 검토 카드의 형식과 허용값 확인


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
# Synthetic assessment summaries, not live sudo policy or a binary catalogue.
awk -F '|' '
 NR==1 {
   if ($0!="case_id|tool_role|reference_listed|context|business_need|approval|scope_fit|telemetry") bad=1
   next
 }
 NF!=8 || $1!~/^R[0-9][0-9]$/ || seen[$1]++ || $2=="" ||
 $3!~/^(yes|no)$/ || $4!~/^(unprivileged|sudo|suid|capabilities|service)$/ ||
 $5!~/^(documented|unknown)$/ || $6!~/^(approved|unknown)$/ ||
 $7!~/^(aligned|review|unknown)$/ || $8!~/^(present|not_collected)$/ {bad=1}
 END {if (NR<2 || bad) exit 2}
' "$COURSE_DATA/tool-review.psv"
printf 'review_schema=valid\n'


### 5. 설정 검토와 실행 자료를 서로 다른 축으로 집계


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
awk -F '|' 'NR>1 {print $1 "|" $3 "|" $7 "|" $8}' \
 "$COURSE_DATA/tool-review.psv" > "$COURSE_OUT/tool-review-results.psv"
awk -F '|' 'NR>1 {scope[$7]++; telemetry[$8]++}
 END {
   printf "aligned=%d review=%d unknown=%d\n", scope["aligned"],scope["review"],scope["unknown"]
   printf "telemetry_present=%d telemetry_not_collected=%d\n",telemetry["present"],telemetry["not_collected"]
 }' "$COURSE_DATA/tool-review.psv" > "$COURSE_OUT/tool-review-counts.txt"
grep -Fx 'aligned=2 review=1 unknown=1' "$COURSE_OUT/tool-review-counts.txt"
grep -Fx 'telemetry_present=2 telemetry_not_collected=2' "$COURSE_OUT/tool-review-counts.txt"
grep -Fx 'R02|yes|unknown|not_collected' "$COURSE_OUT/tool-review-results.psv"
grep -Fx 'R03|no|review|not_collected' "$COURSE_OUT/tool-review-results.psv"
printf 'catalogue_membership_is_not_a_verdict=yes\n'


## GTFOBins 연결

[07-4 전체 해설](../../07-secure-scripting/07-4-gtfobins-review.md)을 읽습니다. tool-review.psv는 가상 검토 카드이며 실제 목록·sudo 정책·사건 증거가 아닙니다. 설정 상태 aligned=2/review=1/unknown=1과 실행 자료 present=2/not_collected=2는 별도 축입니다. R01의 등재만으로 취약, R03의 미등재만으로 안전이라고 결론 내리지 않습니다.


## Red Team ↔ Blue Team 사례 분석

### 사례: UID 0 계정과 쓰기 가능한 업무 파일

**Red Team 질문:** 식별된 계정·권한 예외가 업무 범위를 넘어서는 통제 가능성을 만드는가? UID 0 계정 둘과 그룹 쓰기 가능 파일 하나는 서로 다른 관찰입니다. 두 사실을 연결하는 실행·접근 조건 없이 하나의 권한 상승 경로로 단정하지 않습니다.

| 연결 단계 | 분석 내용 |
|---|---|
| Goal / Boundary | 계정·위임·파일 변경 경계의 과도한 권한 검토 |
| Command / Observation | awk로 UID 0과 권한 요약 읽기, 승인 VM에서는 범위를 정한 find/getcap 조회 |
| System Change / Artifact | 계정·그룹·비트·ACL·Capability·정책의 변경 가능 흔적. 현재 값은 변경 주체를 직접 보여주지 않음 |
| Log prerequisite | 계정 관리·Audit·정책 변경·패키지/배포 기록. sudo 로그는 모든 SUID 실행 기록이 아님 |
| Blue Team Investigation | legacy-admin의 생성·승인·사용 이력, collector 파일의 실행 주체와 수정 권한 비교 |
| Detection | 정상 기준선 밖의 권한 변경과 실제 사용을 별도 경보·조사 상태로 관리 |
| Mitigation | 예외 권한 재검토·업무와 배포 권한 분리·정책 변경 승인. 원본 보존 후 승인 절차로 수정 |

**반례와 해설:** passwd의 SUID는 정상 배포 기준선일 수 있습니다. nologin 설정은 특정 셸 로그인 경로를 제한하지만 UID 0 권한을 없애는 것이 아닙니다. 자동 보고서는 `review`를 `exploited`로 바꾸지 않습니다.

**제출 과제:** 계정 예외와 파일 예외를 별도 항목으로 작성하고, 각각 성립 조건·부족한 자료·정상 반례·완화를 적습니다. 다음 절의 [GTFOBins 검토](../../07-secure-scripting/07-4-gtfobins-review.md)에서 정상 도구의 기능과 실행 권한 문맥을 비교합니다.


## Red Team ↔ Blue Team 사례 분석

### 사례: 업무용 도구 위임의 범위가 문서화되지 않았다

**Red Team 질문:** “파일 보고서 작성이라는 업무에 필요한 권한과 도구가 수행할 수 있는 기능 사이에 차이가 있는가?” 목적은 경계의 과도한 위임 가능성을 식별하는 것입니다. 프로그램 등재만으로 성립 조건이나 영향 범위를 확정하지 않습니다.

| 연결 단계 | 확인·기록할 내용 |
|---|---|
| Technique / Goal | 정상 도구 기능이 의도한 작업 범위를 넘을 가능성이라는 가설 |
| Command / Technique | 제공 권한·업무 요약표를 awk로 읽고, 실제 진단에서는 승인된 설정 사본 검토 |
| Security Meaning | 위임한 업무와 허용 기능이 다르면 최소 권한 검토 필요 |
| System Change / Artifact | 정책 변경 흔적과 도구 실행 흔적은 별개. 파일 읽기는 내용 변경을 남기지 않을 수도 있음 |
| Log / 수집 전제 | sudo 이벤트·승인 이력, 사전 규칙이 있는 Audit/EDR 실행·파일 접근 기록. auth.log가 모든 파일 읽기를 기록하지 않음 |
| Blue Team Investigation | 실행 주체·대상 사용자·인수·파일 범위·시각을 승인 업무와 비교 |
| Detection | 승인 범위와 실행 기록의 차이를 조사 후보로 분류. 도구 이름만으로 경보하지 않음 |
| Mitigation | 업무별 최소 위임, 경로·데이터 접근 통제, 변경 관리, 필요한 감사와 보존을 설계 |

**정상 반례:** 승인된 관리자가 허용된 보고서를 읽었습니다. 이 경우 높은 권한의 실행 기록이 있어도 업무 범위 일치 여부를 먼저 확인해야 합니다.

**자료 부족 사례:** 허용 정책만 있고 실행 기록이 수집되지 않았습니다. “위임 범위 확인 필요, 사용 여부 미확인”까지 보고할 수 있으며 “침해 없음”이나 “권한 상승 성공”은 모두 근거를 넘습니다.

관련 분류는 [Sudo and Sudo Caching — T1548.003](https://attack.mitre.org/techniques/T1548/003/)과 [Setuid and Setgid — T1548.001](https://attack.mitre.org/techniques/T1548/001/)입니다. 정상 정책 검토에 공격 발생 판정을 붙이는 ID가 아닙니다. 실제 행위 문맥과 근거가 맞을 때만 분석 보고서에 연결합니다.

### 탐지 설계 연습

탐지 요구사항을 “GTFOBins 이름 발견 시 경보”가 아니라 “업무 승인 범위와 권한 있는 실행의 불일치 검토”로 작성합니다. 입력에는 호스트·사건 시각·주체·대상 권한·실행 경로·정책 버전·승인 범위가 필요합니다. 사전에 기록되지 않은 필드는 unknown으로 남기고 경보의 확신도를 높이는 근거로 쓰지 않습니다.

평가 데이터는 최소 세 종류를 준비합니다: 승인 업무와 일치, 승인 범위와 차이, 로그 미수집. 각각 정상 검토·추가 검토·판단 보류로 구분되는지 확인합니다. 정책 변경 뒤에는 정상 업무도 재검증하며, 탐지를 통과했다고 최소 권한 설정이 보장되는 것은 아닙니다.


## 역할별 분석 기록

같은 실행 결과로 아래 항목을 작성하고 상대 관점에서 검토합니다. 자동 테스트는 계산과 원본 보존만 확인하며 이 서술 과제는 강사 또는 동료 검토 대상입니다.

| 항목 | 학생 작성 |
|---|---|
| Red Team 목적·필요 조건 | 관찰에서 도출한 질문과 전제 |
| 실제 관찰 | 파일·행·이벤트 ID와 출력 |
| Artifact·로깅 전제 | 확보한 자료와 필요한 기록 기능 |
| Blue Team 조사 | 정상 반례·추가 근거·수집 한계 |
| 탐지·완화 | 필요한 필드·오탐 사례·확인된 원인에 맞는 조치 |


## 기능이 악용되면 어떤 영향이 생기는가

공격자는 정상 프로그램의 기능을 자신에게 허용되지 않은 권한이나 데이터 범위에 적용하려고 합니다. 아래는 **영향과 성립 조건을 설명하는 모델**이며 실행 레시피가 아닙니다. 정상 기능의 실행만으로 아래 공격이 성립하지는 않습니다.

| 기능 | 경계가 잘못 설정됐을 때 가능한 영향 | 추가로 확인해야 하는 조건 | 방어자가 연결할 자료 |
|---|---|---|---|
| 파일 읽기 | 비인가 정보 열람, 민감 자료 노출 | 실제 실행 권한이 대상 파일에 접근할 수 있는가, 그 접근이 업무상 승인됐는가 | 접근 제어·위임 정책·파일 접근 기록·승인 범위 |
| 파일 쓰기 | 무결성 훼손, 설정 변조 | 실제 쓰기 권한과 변경 대상의 보안 역할은 무엇인가 | 변경 전후 해시·메타데이터·승인·후속 동작 |
| 하위 명령 실행 | 단일 작업 위임이 범용 실행 권한으로 확대 | 하위 프로그램 실행 기능, 상속되는 권한, 실행 제약이 어떻게 결합하는가 | 부모/자식 프로세스·실행 권한·인수·승인 |
| 코드·라이브러리 로드 | 신뢰하지 않은 코드가 신뢰된 프로세스 안에서 실행 | 로드 경로와 코드의 변경 주체를 통제하는가 | 배포·파일 변경·프로세스 로드 기록 |
| 자료 송수신 | 비인가 외부 전송이나 승인되지 않은 파일 반입 | 데이터 접근과 통신 경로가 모두 허용되는가 | 목적지·프로세스·전송 기록·자료 반출 승인 |

**Sudo, SUID, Capabilities는 공격 이름이 아니라 권한이 부여되는 서로 다른 문맥**입니다. Sudo는 정책에 따른 실행 위임, SUID는 실행 파일 소유자와 관련된 유효 ID 전환, Capabilities는 세분화된 권한 모델입니다. 실제 적용은 실행 파일 종류, 프로세스 상태, 마운트 옵션 등 조건에 영향을 받습니다. root 소유라는 사실만으로 SUID가 설정됐거나 root로 실행된다고 판단하지 않습니다. 특수 비트나 Capability가 존재한다는 사실만으로 임의 명령 실행이 가능하다고 결론 내리지도 않습니다.


## 확장 실험 Setup

Ubuntu 일반 사용자 환경에서 실행합니다. root로 동작하는 Colab에서는 이 실험을 실행하지 않습니다. sudo·SUID·Capability 부여 없이 자신의 더미 파일만 사용합니다. 앞의 Setup부터 실행한 뒤 아래 셀을 실행하세요. 결과는 COURSE_OUT 안의 새 폴더에 남습니다.


In [ ]:
import os
import platform
import tempfile
if platform.system() != 'Linux' or os.geteuid() == 0:
    raise RuntimeError('이 확장 실험은 Ubuntu 일반 사용자 환경에서 실행하세요.')
os.environ['GTFO_LAB'] = tempfile.mkdtemp(prefix='gtfo-boundary-', dir=os.environ['COURSE_OUT'])
print('GTFO 실습용 임시 폴더 준비 완료')


## GTFOBins 확장 실험 — 권한과 데이터 경계 관찰

목표는 “프로그램이 파일을 읽을 수 있다”와 “그 프로그램이 권한을 상승시켰다”를 분리하는 것입니다. 자신이 소유한 더미 자료로 정상 기능·접근 거부·원본 보존을 관찰하고, 마지막에는 가상 실행 기록을 승인 범위와 비교합니다.

### 1. 환경과 파일의 출처 확인


In [ ]:
%%bash
set -euo pipefail
: "${GTFO_LAB:?실습 Setup부터 실행하세요}"
if [ "$(uname -s)" != Linux ] || [ "$(id -u)" -eq 0 ]; then
    printf 'Ubuntu 일반 사용자 환경에서 실행하세요.\n' >&2
    exit 2
fi
for tool in grep stat sha256sum base64 cmp find awk; do
    command -v "$tool" >/dev/null
done
umask 077
printf 'TRAINING_ONLY\n' > "$GTFO_LAB/report.txt"
printf 'uid=%s\n' "$(id -u)"
TZ=Asia/Seoul date '+observed_at=%Y-%m-%dT%H:%M:%S%:z'
stat -c 'mode=%a owner_uid=%u file=%n' "$GTFO_LAB/report.txt"


출력 예시는 `uid=1000`, `observed_at=...+09:00`, `mode=600 owner_uid=1000 file=/tmp/.../report.txt`입니다. UID·시각·임시 경로는 환경마다 다릅니다. `600`은 소유자 읽기/쓰기만 허용하는 모드입니다. `umask 077`은 이 셀에서 **새로 만드는 파일**의 기본 권한을 제한하며 기존 파일을 소급 변경하지 않습니다.

**Bash Point — 프로세스와 환경:** `command -v`는 필요한 명령의 존재를 확인합니다. `$()`는 표준 출력을 문자열로 받아 인수에 넣습니다. 각 `%%bash` 셀은 별도 프로세스이므로 `umask`와 셸 옵션은 필요한 셀마다 다시 설정합니다.

### 2. 동일한 grep, 다른 파일 접근 결과

먼저 자신이 소유한 자료를 정상적으로 읽습니다. 이어 이 더미 파일의 접근 비트를 잠시 제거하고 같은 작업이 거부되는지 확인합니다. 파일 내용은 바꾸지 않습니다.


In [ ]:
%%bash
set -euo pipefail
: "${GTFO_LAB:?}"
file="$GTFO_LAB/report.txt"
before=$(sha256sum "$file" | cut -d ' ' -f1)
grep -nF 'TRAINING_ONLY' "$file"
trap 'chmod 600 "$file"' EXIT
chmod 000 "$file"
if grep -nF 'TRAINING_ONLY' "$file" > "$GTFO_LAB/read.out" 2> "$GTFO_LAB/read.err"; then
    printf '예상과 달리 읽기 성공: 환경의 추가 권한을 확인하세요.\n' >&2
    exit 1
else
    status=$?
fi
printf 'denied_status=%s\n' "$status"
test "$status" -eq 2
test ! -s "$GTFO_LAB/read.out"
chmod 600 "$file"
trap - EXIT
after=$(sha256sum "$file" | cut -d ' ' -f1)
test "$before" = "$after"
printf 'restored_mode=%s content_unchanged=yes\n' "$(stat -c %a "$file")"


예상 출력:

```text
1:TRAINING_ONLY
denied_status=2
restored_mode=600 content_unchanged=yes
```

**해석:** grep은 같은 기능을 실행했지만 접근 권한이 없으면 읽지 못했습니다. 따라서 GTFOBins에 파일 읽기 기능이 설명되어 있다는 이유만으로 접근 제어가 자동 우회되는 것은 아닙니다. 이 파일은 학생 자신의 파일이므로 학생이 권한을 원복한 것 역시 권한 상승이 아닙니다. `chmod`는 메타데이터를 바꾸므로 이 실험은 실제 증거 원본에 수행하지 않습니다. 해시가 같아도 메타데이터까지 보존됐다는 뜻은 아닙니다.

**실패 사례:** `grep ... || true`로 모든 오류를 숨기면 검색어 부재(1)와 읽기 오류(2)를 구분하지 못합니다. 위 코드는 `else`에 들어가자마자 `$?`를 저장하고 기대한 실패인지 검사합니다. 특수 권한이 있는 환경에서 성공한다면 권한을 더 부여해 맞추지 말고 일반 사용자 환경으로 돌아갑니다. EXIT trap도 SIGKILL이나 호스트 종료까지 보장하지 않습니다.

### 3. 파일 쓰기 기능과 무결성 관찰


In [ ]:
%%bash
set -euo pipefail
: "${GTFO_LAB:?}"
umask 077
cp "$GTFO_LAB/report.txt" "$GTFO_LAB/work-copy.txt"
printf 'REVIEW_NOTE\n' >> "$GTFO_LAB/work-copy.txt"
if cmp -s "$GTFO_LAB/report.txt" "$GTFO_LAB/work-copy.txt"; then
    printf '사본 변경이 관찰되지 않았습니다.\n' >&2
    exit 1
else
    status=$?
fi
test "$status" -eq 1
printf 'original_lines=%s copy_lines=%s\n' \
    "$(wc -l < "$GTFO_LAB/report.txt" | tr -d ' ')" \
    "$(wc -l < "$GTFO_LAB/work-copy.txt" | tr -d ' ')"
sha256sum "$GTFO_LAB/report.txt" "$GTFO_LAB/work-copy.txt"


`original_lines=1 copy_lines=2`이며 두 해시는 다릅니다. 이것은 학생이 자신의 **작업 사본에 메모를 추가한 정상 변경**입니다. 파일 쓰기 능력과 설정 변조 공격을 같다고 보지 않습니다. 실제 조사에서는 변경 대상의 역할, 변경 주체의 권한, 승인 기록, 후속 실행을 연결해야 합니다. 해시 차이는 바이트 차이의 근거이지 공격자 신원의 근거가 아닙니다.

**Bash Point — 리다이렉션:** `>>`는 셸이 출력 파일을 열어 내용을 덧붙입니다. 단순 파일 출력을 “명령 자체가 모든 쓰기를 수행했다”고 해석하면 프로세스·파일 접근 기록을 잘못 연결할 수 있습니다.

### 4. 인코딩은 기밀성 보호가 아니다


In [ ]:
%%bash
set -euo pipefail
: "${GTFO_LAB:?}"
umask 077
base64 "$GTFO_LAB/report.txt" > "$GTFO_LAB/report.b64"
base64 --decode "$GTFO_LAB/report.b64" > "$GTFO_LAB/decoded.txt"
cmp -s "$GTFO_LAB/report.txt" "$GTFO_LAB/decoded.txt"
printf 'roundtrip_equal=yes\n'
stat -c 'encoded_mode=%a' "$GTFO_LAB/report.b64"


예상 결과는 `roundtrip_equal=yes`, `encoded_mode=600`입니다. 더미 자료를 변환했다가 복원했을 뿐이고 권한 변경이나 외부 전송은 없습니다. base64는 암호화가 아니므로 문자열이 바로 읽히지 않는다는 이유로 비밀 자료를 공개해도 되는 것은 아닙니다. 생성한 인코딩 파일과 복원 파일에도 원본과 같은 취급 기준이 필요합니다. 도구 실행이나 인코딩 문자열 하나만으로 정보 유출 발생을 확정하지 않습니다.

### 5. 제한된 범위의 권한 메타데이터 확인


In [ ]:
%%bash
set -euo pipefail
: "${GTFO_LAB:?}"
find "$GTFO_LAB" -maxdepth 1 -type f -perm -4000 -print > "$GTFO_LAB/suid-list.txt"
test ! -s "$GTFO_LAB/suid-list.txt"
printf 'lab_suid_files=0\n'
stat -c 'mode=%a owner_uid=%u' "$GTFO_LAB/report.txt"
if command -v getcap >/dev/null; then
    getcap "$GTFO_LAB/report.txt"
    printf 'getcap_checked=yes\n'
else
    printf 'getcap_checked=no (선택 도구 미설치)\n'
fi


`lab_suid_files=0`은 **이 임시 폴더의 일반 파일**에서 SUID 비트를 발견하지 않았다는 뜻입니다. 호스트 전체 안전 판정이 아닙니다. getcap의 빈 출력은 성공적으로 확인한 이 파일에 표시할 파일 Capability가 없다는 뜻이며, 호출 실패나 권한 부족과 구분합니다. 이 실습에서 특수 권한은 부여하지 않습니다. 07-3의 SUID/Capability 설명과 실제 파일 메타데이터의 역할을 연결하는 관찰입니다.

### 6. 실행 기록과 승인 범위를 비교하기

다음 자료는 **교육용으로 작성한 정규화 이벤트 세 개**입니다. 방금 실행한 명령의 OS 감사 로그가 아니며 실제 sudo/audit 로그 형식도 아닙니다. 가상의 승인 업무는 “analyst가 자신의 권한으로 report 범주의 자료를 읽는 것”입니다.


In [ ]:
%%bash
set -euo pipefail
: "${GTFO_LAB:?}"
umask 077
printf '%s\n' \
  'id|time|actor|effective_role|action|resource' \
  'E01|2026-09-12T09:00:00+09:00|analyst|analyst|read|report' \
  'E02|2026-09-12T09:01:00+09:00|analyst|administrator|read|outside_scope' \
  'E03|2026-09-12T09:02:00+09:00|analyst|unknown|read|report' \
  > "$GTFO_LAB/events.psv"
awk -F '|' 'NR>1 {
  if ($4=="unknown") verdict="unknown";
  else if ($3=="analyst" && $4=="analyst" && $5=="read" && $6=="report") verdict="aligned";
  else verdict="review";
  print $1, verdict
}' "$GTFO_LAB/events.psv" | tee "$GTFO_LAB/verdicts.txt"
test "$(grep -c ' aligned$' "$GTFO_LAB/verdicts.txt")" -eq 1
test "$(grep -c ' review$' "$GTFO_LAB/verdicts.txt")" -eq 1
test "$(grep -c ' unknown$' "$GTFO_LAB/verdicts.txt")" -eq 1


```text
E01 aligned
E02 review
E03 unknown
```

E02는 가상의 실행 권한·자료 범위가 승인 예시와 달라 **추가 검토할 항목**입니다. 실제 공격 재현 결과나 확정 침해가 아닙니다. 임시 관리자 작업 승인 여부라는 정상 반례를 확인해야 합니다. E03은 권한 자료가 없으므로 E01과 합쳐 정상으로 집계하지 않습니다. 이 짧은 awk는 형식이 고정된 세 행의 설명용 분류기이며, 신뢰하지 않은 실제 로그용 검증기나 sudoers 파서가 아닙니다.

**Artifact와 탐지의 한계:** 2단계의 `read.err`, 6단계의 `verdicts.txt`는 실습 프로그램이 만든 파일입니다. auth.log/secure가 모든 파일 읽기를 기록하는 것은 아닙니다. 실제 추적에는 사전에 설정된 감사 규칙·EDR 수집, 명령 실행 문맥, 파일 대상, 승인 자료가 필요하며 수집하지 않은 필드는 추측하지 않습니다.

### 확장 실습 완료 기준

- `denied_status=2`, 권한 원복 600, 원본 내용 보존을 설명한다.
- 파일 소유자·유효 권한·rwx·SUID·Capability를 같은 개념으로 쓰지 않는다.
- 사본의 변경과 원본의 변경, 파일 해시와 메타데이터 보존을 구분한다.
- base64 왕복 일치가 암호화나 비인가 접근 성공을 뜻하지 않는 이유를 설명한다.
- E01/E02/E03을 일치/추가 검토/미확인으로 나누고 정상 반례를 제시한다.
- 각 기능별로 **공격자가 얻으려는 효과 / 필요한 권한 조건 / 실제 관찰 / 미수집 자료 / 완화안**을 한 행씩 작성한다.

이 실험의 PASS는 정상 기능과 접근 제어를 이해했다는 뜻입니다. sudo·SUID 악용이나 실제 권한 상승을 실행·검증했다는 뜻이 아닙니다. 다음 검토에서는 업무별 권한을 최소화하고 자료 범위·정책 변경·감사 설정을 함께 확인합니다.


## Checks

각 STEP의 test는 고정 자료의 계산 결과를 검사합니다. 아래는 원본 내용 보존을 확인합니다. 실행 성공과 침해 판정은 다릅니다. 어떤 결과가 사실이고 어떤 결론이 가설인지 교안 질문에 답합니다.


In [ ]:
after = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
assert before == after
print('원본 내용 보존: PASS')
print('분석 결과 파일 수:', sum(p.is_file() for p in output.rglob('*')))


## Next Steps

교안의 완료 기준에 따라 근거·정상 행위 가능성·누락·추가 확인을 제출합니다. 결과는 검토용 임시 폴더에 남습니다. 재실행은 Setup부터 새 폴더에서 시작하며 실제 증거를 공개 저장소에 올리지 않습니다.
